In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, FloatType, TimestampType

catalog = 'Ecommerce_BI'

In [0]:
df_bronze = spark.table(f'{catalog}.bronze.brz_brands')
df_bronze.show(10)

##Brands

###Trimming Brand Name

In [0]:
df_silver = df_bronze.withColumn("brand_name", F.trim(F.col("brand_name")))
df_silver.show(10)

###Removing Special characters from brand code

In [0]:
df_silver = df_silver.withColumn("brand_code", F.regexp_replace(F.col("brand_code"), r'[^A-Za-z0-9]', ''))
df_silver.show(10)

###Removing redundancies and annomalies from category code

In [0]:
df_silver.select("category_code").distinct().show()

In [0]:
redundencies = {
    "BOOKS" : "BKS",
    "GROCERY" : "GRCY",
    "TOYS" : "TOY"
}

df_silver = df_silver.replace(redundencies, subset="category_code")
df_silver.select("category_code").distinct().show()

###Writing the table into the silver layer

In [0]:
display(df_silver)
df_silver.write.format('delta').mode('overwrite').option('mergeSchema', 'true').saveAsTable(f'{catalog}.silver.slvr_brands')

###Category

In [0]:
df_bronze = spark.table(f'{catalog}.bronze.brz_categories')
df_bronze.show(10)

###Let's investigate duplicates in the primary key

In [0]:
df_duplicates = df_bronze.groupBy("category_code").count().filter(F.col("count") > 1)
display(df_duplicates)

In [0]:
df_silver = df_bronze.dropDuplicates(["category_code"])
display(df_silver)

### Nomalizing category code to upper case

In [0]:
df_silver = df_silver.withColumn("category_code", F.upper(F.col("category_code")))
display(df_silver)

In [0]:
df_silver.write.format('delta').mode('overwrite').option('mergeSchema', 'True').saveAsTable(f'{catalog}.silver.slvr_categories')

##Products

###holder

In [0]:
df_bronze = spark.table(f'{catalog}.bronze.brz_products')
df_bronze.show(10)

raw_count, column_count = df_bronze.count(), len(df_bronze.columns)
print("Raw count " + str(raw_count))
print("column count " + str(column_count))

###Remove the mesuring unit from weight

In [0]:
df_silver = df_bronze.withColumn("weight_grams", F.regexp_replace(F.col("weight_grams"), "g","").cast(FloatType()))

df_silver.show(5, truncate=False)

### Fixing the formatting on the lenght column

In [0]:
df_silver.select("length_cm").show(3)

In [0]:
df_silver = df_silver.withColumn("length_cm", F.regexp_replace(F.col("length_cm"), ",", "."))
df_silver.show(5, truncate=False)

###We normalize brand and category code to uppercase

In [0]:
df_silver = df_silver.withColumn("brand_code", F.upper(F.col("brand_code"))).withColumn("category_code", F.upper(F.col("category_code")))
df_silver.show(5, truncate=False)

###Investigating the material column 

In [0]:
df_silver.select("material").distinct().show()

In [0]:
#Fixing identified spelling mistakes
df_silver = df_silver.withColumn(
    "material",
    F.when(F.col("material") == "Coton", "Cotton")
     .when(F.col("material") == "Alumium", "Aluminum")
     .when(F.col("material") == "Ruber", "Rubber")
     .otherwise(F.col("material"))
)

df_silver.select("material").distinct().show()

###Investigating rating_count

In [0]:
df_silver.filter(F.col("rating_count") < 0).select("rating_count").show(5)

In [0]:
#Fixing negative values for rating_count
df_silver = df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
    .otherwise(F.lit(0)) #Replacing Null Values with 0
)

### Final check

In [0]:
df_silver.select("brand_code", "category_code", "length_cm", "weight_grams", "size", "color", "material", "rating_count").show(10, False)

In [0]:
df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog}.silver.slvr_products")

## Customers

In [0]:
df_bronze = spark.table(f"{catalog}.bronze.brz_customers")

raw_count, column_count = df_bronze.count(), len(df_bronze.columns)
print(f"Raw Data: {raw_count:,} records, {column_count} columns")

In [0]:
df_bronze.show()

### cheking null values on the primary key

In [0]:
null_count = df_bronze.filter(F.col("customer_id").isNull()).count()
null_count

In [0]:
#With only 300 null records we can drop the null values
df_silver = df_bronze.dropna(subset=["customer_id"])

row_count = df_silver.count()
print(f"Number of records in silver table: {row_count}")

###Some phone Numbers are null making them unavailable 

In [0]:
df_silver = df_silver.fillna("Unavailable", subset=["phone"])

df_silver.filter(F.col("phone").isNull()).show()
df_silver.filter(F.col("phone") == "Unavailable").show(5)

In [0]:
df_silver.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"{catalog}.silver.slvr_customers")

##Date

In [0]:
df_bronze = spark.table(f"{catalog}.bronze.brz_date")

row_count, column_count = df_bronze.count(), len(df_bronze.columns)
print(f"Raw Data: {row_count:,} records, {column_count} columns")


df_bronze.show(5)

df_bronze.printSchema()

###Converting the date into DateType

In [0]:
from pyspark.sql.functions import to_date
df_silver = df_bronze.withColumn("date", to_date(df_bronze["date"], "dd-MM-yyyy"))

df_silver.show(5)

df_silver.printSchema()

###Checking for dupelicate dates

In [0]:
df_duplicates = df_silver.groupBy("date").count().filter("count > 1")
df_duplicates.show()

In [0]:
df_silver = df_silver.dropDuplicates(["date"])
df_duplicates = df_silver.groupBy("date").count().filter("count > 1")
df_duplicates.show()

###Normalizing day_name

In [0]:
df_silver.select("day_name").distinct().show()

In [0]:
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))
df_silver.select("day_name").distinct().show()

###Fixing Negative values on week_of_year

In [0]:
df_silver = df_silver.withColumn("week_of_year",
            F.when(F.col("week_of_year").isNotNull(), F.abs(F.col("week_of_year")))
            .otherwise("Unavailable")
            )
df_silver.show(5)

###Enhancing `quarter` and `week_of_year`

In [0]:
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year")))

df_silver = df_silver.withColumn("week_of_year", F.concat_ws("", F.lit("week"), F.col("week_of_year"), F.lit("-"), F.col("year")))

df_silver.show(5)

In [0]:
df_silver = df_silver.withColumnRenamed("week_of_year", "week")


df_silver.write.format('delta').mode('overwrite').option("mergeSchema", True).saveAsTable(f"{catalog}.silver.slvr_date")